# Лабораторные работы 1–5

В ноутбуке последовательно выполнены лабораторные работы по KNN, логистической и линейной регрессии, решающему дереву, случайному лесу и градиентному бустингу. Для каждой работы приведены исходный sklearn-baseline, проверка гипотезы/улучшение, самостоятельная реализация, оценка и сравнение. В конце результаты сохраняются в `results/`; отдельный ноутбук `final_comparison.ipynb` читает эти файлы и сводит все алгоритмы.


## Общая постановка, данные и метрики

### Наборы данных и практический смысл

- **Классификация — `sklearn.datasets.load_digits` (Digits).** Каждый объект описан 64 числовыми признаками — интенсивностями пикселей изображения 8×8; целевая переменная принимает одно из десяти значений (цифры 0–9). Это учебная постановка распознавания рукописных цифр, связанная с OCR и автоматизацией ввода документов.
- **Регрессия — `sklearn.datasets.load_diabetes` (Diabetes).** Используются 10 количественных признаков и непрерывная целевая переменная — количественный показатель прогрессирования заболевания. Это задача прогнозирования числового показателя, а не постановка бинарной диагностики. Практический аналог — поддержка количественной оценки динамики состояния; модель не предназначена для самостоятельного клинического решения.

### Метрики и критерии выбора

Для классификации считаются **Accuracy**, **macro-Precision**, **macro-Recall** и **macro-F1**. Accuracy показывает общую долю правильных ответов, а macro-метрики сначала рассчитываются отдельно для каждого класса и затем усредняются с одинаковым весом. Поэтому они не позволяют хорошему качеству на часто встречающихся цифрах полностью скрыть ошибки на редких классах. Основной критерий ранжирования — macro-F1.

Для регрессии используются **MAE**, **RMSE** и **R²**. MAE показывает средний абсолютный промах в единицах целевой переменной; RMSE сильнее штрафует крупные ошибки; R² сопоставляет модель с прогнозом среднего значения и показывает долю объяснённой вариации. Основной критерий ранжирования — минимальный RMSE, MAE и R² используются как дополнительные проверки.

### Единый протокол эксперимента

Для всех пяти работ используются одинаковые обучающие и тестовые объекты: 80/20, `random_state=42`; для Digits разбиение стратифицировано. Подбор параметров выполняется через 5-fold CV только на обучающей части. Масштабирование встроено в `Pipeline`, поэтому статистики масштабирования оцениваются внутри каждого CV-фолда. Тестовая часть используется только для итоговой оценки. Все результаты относятся к одному фиксированному разбиению и не заменяют проверку устойчивости на нескольких разбиениях.


## Общий технический блок

Здесь находятся только импорты, определения собственных алгоритмов, функции расчёта метрик и единая подготовка выборок. Обучение конкретных моделей расположено ниже, непосредственно под пунктами соответствующих лабораторных.

In [1]:
# Импортируем библиотеки, определяем классы и готовим общие выборки.
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import load_digits, load_diabetes
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold, KFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             mean_absolute_error, mean_squared_error, r2_score,
                             confusion_matrix, ConfusionMatrixDisplay)
from sklearn.neighbors import KNeighborsClassifier, KNeighborsRegressor
from sklearn.linear_model import LogisticRegression, LinearRegression, Ridge
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor, HistGradientBoostingClassifier, HistGradientBoostingRegressor

ROOT = Path.cwd()
OUT = ROOT / 'results'; FIG = OUT / 'figures'
OUT.mkdir(exist_ok=True); FIG.mkdir(exist_ok=True)
RANDOM_STATE = 42

# Определяем собственные модели и сохраняем параметры обучения.
class MyKNNClassifier:
    """KNN-классификатор на NumPy: расстояния и голосование большинства."""
    def __init__(self, n_neighbors=5, scale=True): self.n_neighbors, self.scale = n_neighbors, scale
    def fit(self, X, y):
        X = np.asarray(X, float); self.mean_ = X.mean(0); self.std_ = X.std(0); self.std_[self.std_ == 0] = 1
        self.X_ = (X-self.mean_)/self.std_ if self.scale else X; self.y_ = np.asarray(y); return self
    def predict(self, X):
        X = np.asarray(X, float); X = (X-self.mean_)/self.std_ if self.scale else X; result=[]
        for row in X:
            d=np.sqrt(((self.X_-row)**2).sum(1)); ids=np.argsort(d)[:self.n_neighbors]
            labels,counts=np.unique(self.y_[ids],return_counts=True); result.append(labels[np.argmax(counts)])
        return np.asarray(result)

class MyKNNRegressor:
    """KNN-регрессор на NumPy с масштабированием, сохранённым внутри fit."""
    def __init__(self, n_neighbors=5, scale=True): self.n_neighbors, self.scale = n_neighbors, scale
    def fit(self,X,y):
        X=np.asarray(X,float); self.mean_=X.mean(0); self.std_=X.std(0); self.std_[self.std_==0]=1
        self.X_=(X-self.mean_)/self.std_ if self.scale else X; self.y_=np.asarray(y,float); return self
    def predict(self,X):
        X=np.asarray(X,float); X=(X-self.mean_)/self.std_ if self.scale else X; result=[]
        for row in X:
            d=np.sqrt(((self.X_-row)**2).sum(1)); ids=np.argsort(d)[:self.n_neighbors]; result.append(self.y_[ids].mean())
        return np.asarray(result)

class MyLinearRegression:
    """Линейная регрессия через псевдообратную матрицу."""
    def fit(self,X,y):
        X=np.asarray(X,float); self.mean_=X.mean(0); self.std_=X.std(0); self.std_[self.std_==0]=1
        Z=(X-self.mean_)/self.std_; Z=np.c_[np.ones(len(Z)),Z]; self.coef_=np.linalg.pinv(Z.T@Z)@Z.T@np.asarray(y,float); self.loss_history_=[float(np.mean((Z@self.coef_-y)**2))]; return self
    def predict(self,X):
        Z=(np.asarray(X,float)-self.mean_)/self.std_; return np.c_[np.ones(len(Z)),Z]@self.coef_

class MyLogisticRegression:
    """Многоклассовая softmax-регрессия с L2-регуляризацией и early stopping."""
    def __init__(self,lr=.05,epochs=1500,reg=.001,tolerance=1e-7,patience=30): self.lr,self.epochs,self.reg,self.tolerance,self.patience=lr,epochs,reg,tolerance,patience
    def fit(self,X,y):
        X=np.asarray(X,float); self.mean_=X.mean(0); self.std_=X.std(0); self.std_[self.std_==0]=1
        Z=(X-self.mean_)/self.std_; Z=np.c_[np.ones(len(Z)),Z]; self.classes_=np.unique(y); Y=np.eye(len(self.classes_))[np.searchsorted(self.classes_,y)]; self.W=np.zeros((Z.shape[1],len(self.classes_))); self.loss_history_=[]; best=np.inf; stale=0
        for _ in range(self.epochs):
            s=Z@self.W; s-=s.max(1,keepdims=True); P=np.exp(s); P/=P.sum(1,keepdims=True); loss=-np.mean(np.sum(Y*np.log(P+1e-12),1)) + self.reg*np.sum(self.W[1:]**2)/2
            g=Z.T@(P-Y)/len(Z); g[1:]+=self.reg*self.W[1:]; self.W-=self.lr*g; self.loss_history_.append(float(loss))
            if best-loss > self.tolerance: best,stale=loss,0
            else: stale += 1
            if stale >= self.patience: break
        return self
    def predict(self,X):
        Z=(np.asarray(X,float)-self.mean_)/self.std_; return self.classes_[(np.c_[np.ones(len(Z)),Z]@self.W).argmax(1)]


class MyDecisionTree:
    """Собственное дерево CART с критерием Джини и суммой квадратов ошибок."""
    def __init__(self, task='classification', max_depth=5, min_samples_leaf=2, random_state=42):
        self.task, self.max_depth, self.min_samples_leaf, self.random_state = task, max_depth, min_samples_leaf, random_state
    def _impurity(self, y):
        if self.task == 'regression': return float(np.sum((y-y.mean())**2))
        _, counts = np.unique(y, return_counts=True); p = counts / len(y)
        return float(len(y) * (1 - np.sum(p*p)))
    def _build(self, X, y, depth):
        labels, counts = np.unique(y, return_counts=True)
        value = float(np.mean(y)) if self.task == 'regression' else labels[np.argmax(counts)]
        node = {'value': value}
        if (self.max_depth is not None and depth >= self.max_depth) or len(y) < 2*self.min_samples_leaf or len(labels) == 1: return node
        best = (0.0, None, None); parent = self._impurity(y)
        for j in self._rng.choice(X.shape[1], min(8,X.shape[1]), replace=False):
            vals = np.unique(X[:,j])
            thresholds=(vals[:-1] + vals[1:]) / 2
            if len(thresholds)>5: thresholds=thresholds[np.linspace(0,len(thresholds)-1,5,dtype=int)]
            for threshold in thresholds:
                mask = X[:,j] <= threshold; nl, nr = mask.sum(), (~mask).sum()
                if nl < self.min_samples_leaf or nr < self.min_samples_leaf: continue
                gain = parent-self._impurity(y[mask])-self._impurity(y[~mask])
                if gain > best[0]: best = (gain, j, threshold)
        if best[1] is None: return node
        j,t=best[1],best[2]; mask=X[:,j] <= t
        node.update(feature=j, threshold=float(t), left=self._build(X[mask],y[mask],depth+1), right=self._build(X[~mask],y[~mask],depth+1))
        return node
    def fit(self, X, y):
        self._rng = np.random.default_rng(self.random_state)
        self.root_=self._build(np.asarray(X,float),np.asarray(y),0); return self
    def _predict_one(self, row):
        n=self.root_
        while 'feature' in n: n=n['left'] if row[n['feature']] <= n['threshold'] else n['right']
        return n['value']
    def predict(self, X): return np.asarray([self._predict_one(row) for row in np.asarray(X,float)])

class MyRandomForest:
    """Ансамбль собственных CART-деревьев на bootstrap-выборках."""
    def __init__(self, task='classification', n_estimators=15, max_depth=5, min_samples_leaf=2, random_state=42):
        self.task,self.n_estimators,self.max_depth,self.min_samples_leaf,self.random_state=task,n_estimators,max_depth,min_samples_leaf,random_state
    def fit(self,X,y):
        X=np.asarray(X,float); y=np.asarray(y); rng=np.random.default_rng(self.random_state); self.trees_=[]
        for _ in range(self.n_estimators):
            ids=rng.integers(0,len(y),len(y))
            tree_seed=int(rng.integers(0, 2**31-1))
            self.trees_.append(MyDecisionTree(self.task,self.max_depth,self.min_samples_leaf,tree_seed).fit(X[ids],y[ids]))
        return self
    def predict(self,X):
        p=np.asarray([t.predict(X) for t in self.trees_])
        if self.task=='regression': return p.mean(axis=0)
        return np.asarray([np.unique(col,return_counts=True)[0][np.argmax(np.unique(col,return_counts=True)[1])] for col in p.T])

class MyGradientBoosting:
    """Собственный градиентный бустинг: MSE для регрессии и softmax-cross-entropy для классификации."""
    def __init__(self, task='regression', n_estimators=15, learning_rate=.1, max_depth=2, random_state=42):
        self.task, self.n_estimators, self.learning_rate, self.max_depth, self.random_state = task, n_estimators, learning_rate, max_depth, random_state

    @staticmethod
    def _softmax(scores):
        z = scores - scores.max(axis=1, keepdims=True)
        e = np.exp(z)
        return e / e.sum(axis=1, keepdims=True)

    def fit(self, X, y):
        X = np.asarray(X, float); y = np.asarray(y)
        rng = np.random.default_rng(self.random_state)
        if self.task == 'regression':
            self.initial_ = float(np.mean(y))
            pred = np.full(len(y), self.initial_, dtype=float)
            self.trees_ = []
            for _ in range(self.n_estimators):
                residual = y.astype(float) - pred
                tree = MyDecisionTree('regression', self.max_depth, 2, int(rng.integers(0, 2**31-1))).fit(X, residual)
                pred += self.learning_rate * tree.predict(X)
                self.trees_.append(tree)
            return self

        self.classes_ = np.unique(y)
        Y = np.eye(len(self.classes_))[np.searchsorted(self.classes_, y)]
        prior = np.clip(Y.mean(axis=0), 1e-8, 1.0)
        self.initial_ = np.log(prior)
        scores = np.tile(self.initial_, (len(y), 1))
        self.trees_ = [[] for _ in self.classes_]
        # Каждый раунд обучает по одному регрессионному дереву на отрицательный градиент для каждого класса.
        for _ in range(self.n_estimators):
            probabilities = self._softmax(scores)
            updates = []
            for k in range(len(self.classes_)):
                residual = Y[:, k] - probabilities[:, k]
                tree = MyDecisionTree('regression', self.max_depth, 2, int(rng.integers(0, 2**31-1))).fit(X, residual)
                updates.append(tree.predict(X))
                self.trees_[k].append(tree)
            scores += self.learning_rate * np.column_stack(updates)
        return self

    def predict(self, X):
        X = np.asarray(X, float)
        if self.task == 'regression':
            out = np.full(len(X), self.initial_, dtype=float)
            for tree in self.trees_:
                out += self.learning_rate * tree.predict(X)
            return out
        scores = np.tile(self.initial_, (len(X), 1))
        for k, trees in enumerate(self.trees_):
            for tree in trees:
                scores[:, k] += self.learning_rate * tree.predict(X)
        return self.classes_[scores.argmax(axis=1)]

def cls_metrics(model,Xtr,ytr,Xte,yte,name,lab,implementation='sklearn',stage='baseline'):
    model.fit(Xtr,ytr); p=model.predict(Xte)
    return {'lab':lab,'task':'classification','model':name,'implementation':implementation,'stage':stage,'accuracy':accuracy_score(yte,p),'precision_macro':precision_score(yte,p,average='macro',zero_division=0),'recall_macro':recall_score(yte,p,average='macro',zero_division=0),'f1_macro':f1_score(yte,p,average='macro',zero_division=0)}, p, model

def reg_metrics(model,Xtr,ytr,Xte,yte,name,lab,implementation='sklearn',stage='baseline'):
    model.fit(Xtr,ytr); p=model.predict(Xte)
    return {'lab':lab,'task':'regression','model':name,'implementation':implementation,'stage':stage,'MAE':mean_absolute_error(yte,p),'RMSE':mean_squared_error(yte,p)**.5,'R2':r2_score(yte,p)}, p, model

def add_cls(result, rows, preds, models, Xtr,ytr,Xte,yte,name,lab,implementation='sklearn',stage='baseline'):
    row,p,m=cls_metrics(result,Xtr,ytr,Xte,yte,name,lab,implementation,stage); rows.append(row); preds[f'{lab}_classification_{name}_{stage}']=p; models[f'{lab}_classification_{name}_{stage}']=m

def add_reg(result, rows, preds, models, Xtr,ytr,Xte,yte,name,lab,implementation='sklearn',stage='baseline'):
    row,p,m=reg_metrics(result,Xtr,ytr,Xte,yte,name,lab,implementation,stage); rows.append(row); preds[f'{lab}_regression_{name}_{stage}']=p; models[f'{lab}_regression_{name}_{stage}']=m

dig=load_digits(as_frame=True); Xc,yc=dig.data,dig.target
Xc_tr,Xc_te,yc_tr,yc_te=train_test_split(Xc,yc,test_size=.2,stratify=yc,random_state=RANDOM_STATE)
dia=load_diabetes(as_frame=True); Xr,yr=dia.data,dia.target
Xr_tr,Xr_te,yr_tr,yr_te=train_test_split(Xr,yr,test_size=.2,random_state=RANDOM_STATE)

# Добавляем агрегаты яркости изображения и взаимодействия медицинских признаков.
def digit_features(df):
    out=df.copy(); out['pixel_mean']=df.mean(axis=1); out['pixel_std']=df.std(axis=1); out['active_pixels']=(df>0).sum(axis=1); return out
def diabetes_features(df):
    out=df.copy(); out['bmi_bp_interaction']=df['bmi']*df['bp']; out['bmi_squared']=df['bmi']**2; out['s5_squared']=df['s5']**2; return out
Xcfe=digit_features(Xc); Xcfe_tr,Xcfe_te=Xcfe.loc[Xc_tr.index],Xcfe.loc[Xc_te.index]
Xrfe=diabetes_features(Xr); Xrfe_tr,Xrfe_te=Xrfe.loc[Xr_tr.index],Xrfe.loc[Xr_te.index]
rows=[]; predictions={}; fitted={}
best_params = {}  # Параметры CV сохраняются отдельно по каждой лабораторной и задаче.


# Лабораторная работа №1 — метод ближайших соседей

## Пункт 1. Подготовка данных и метрики

Используем Digits для классификации и Diabetes для регрессии. Для KNN принципиален масштаб признаков, потому что решение основано на расстояниях.

In [2]:
# Выполняем подготовку данных и показываем её результат.
print('Digits: обучение', Xc_tr.shape, 'тест', Xc_te.shape)
print('Diabetes: обучение', Xr_tr.shape, 'тест', Xr_te.shape)
print('Классов Digits:', yc.nunique(), 'признаков Digits:', Xc.shape[1], 'признаков Diabetes:', Xr.shape[1])


Digits: обучение (1437, 64) тест (360, 64)
Diabetes: обучение (353, 10) тест (89, 10)
Классов Digits: 10 признаков Digits: 64 признаков Diabetes: 10


## Пункт 2. Исходная модель и оценка

Сначала обучаем KNN с пятью соседями для классификации и семью соседями для регрессии без подбора параметров. Одновременно сохраняем вариант со стандартизацией, чтобы увидеть влияние масштаба.

In [3]:
# Обучаем исходные KNN и сразу выводим их метрики.
add_cls(KNeighborsClassifier(5),rows,predictions,fitted,Xc_tr,yc_tr,Xc_te,yc_te,'KNN без масштабирования','ЛР1','sklearn','baseline')
add_reg(KNeighborsRegressor(7),rows,predictions,fitted,Xr_tr,yr_tr,Xr_te,yr_te,'KNN без масштабирования','ЛР1','sklearn','baseline')
add_cls(Pipeline([('scale',StandardScaler()),('model',KNeighborsClassifier(5))]),rows,predictions,fitted,Xc_tr,yc_tr,Xc_te,yc_te,'KNN со стандартизацией','ЛР1','sklearn','baseline_scaled')
add_reg(Pipeline([('scale',StandardScaler()),('model',KNeighborsRegressor(7))]),rows,predictions,fitted,Xr_tr,yr_tr,Xr_te,yr_te,'KNN со стандартизацией','ЛР1','sklearn','baseline_scaled')
display(pd.DataFrame(rows)[pd.DataFrame(rows).lab.eq('ЛР1')])


,lab,task,model,implementation,stage,accuracy,precision_macro,recall_macro,f1_macro,MAE,RMSE,R2
0,ЛР1,classification,KNN без масштабирования,sklearn,baseline,0.983333,0.983449,0.983095,0.983046,NaN,NaN,NaN
1,ЛР1,regression,KNN без масштабирования,sklearn,baseline,NaN,NaN,NaN,NaN,42.648475,54.653225,0.436223
2,ЛР1,classification,KNN со стандартизацией,sklearn,baseline_scaled,0.963889,0.964590,0.963722,0.963440,NaN,NaN,NaN
3,ЛР1,regression,KNN со стандартизацией,sklearn,baseline_scaled,NaN,NaN,NaN,NaN,42.781701,54.493337,0.439517


## Пункт 3. Гипотеза и улучшенный вариант

Гипотеза: подбор числа соседей и стандартизация могут улучшить качество. Число соседей выбираем кросс-валидацией только на обучающей части, затем сравниваем настроенный KNN с исходной моделью.

In [4]:
# Подбираем число соседей и проверяем настроенный вариант со стандартизацией.
cv_c=StratifiedKFold(5,shuffle=True,random_state=RANDOM_STATE); cv_r=KFold(5,shuffle=True,random_state=RANDOM_STATE)
gc=GridSearchCV(KNeighborsClassifier(),{'n_neighbors':[3,5,7,11]},cv=cv_c,scoring='f1_macro').fit(Xc_tr,yc_tr)
gr=GridSearchCV(KNeighborsRegressor(),{'n_neighbors':[3,5,7,11]},cv=cv_r,scoring='neg_root_mean_squared_error').fit(Xr_tr,yr_tr)
add_cls(gc.best_estimator_,rows,predictions,fitted,Xc_tr,yc_tr,Xc_te,yc_te,'KNN настроенный','ЛР1','sklearn','tuned')
add_reg(gr.best_estimator_,rows,predictions,fitted,Xr_tr,yr_tr,Xr_te,yr_te,'KNN настроенный','ЛР1','sklearn','tuned')
add_cls(Pipeline([('scale',StandardScaler()),('model',KNeighborsClassifier(gc.best_params_['n_neighbors']))]),rows,predictions,fitted,Xc_tr,yc_tr,Xc_te,yc_te,'KNN настроенный + стандартизация','ЛР1','sklearn','improved_sklearn')
add_reg(Pipeline([('scale',StandardScaler()),('model',KNeighborsRegressor(gr.best_params_['n_neighbors']))]),rows,predictions,fitted,Xr_tr,yr_tr,Xr_te,yr_te,'KNN настроенный + стандартизация','ЛР1','sklearn','improved_sklearn')
best_params['ЛР1'] = {'classification': gc.best_params_, 'regression': gr.best_params_}
print('Лучшие параметры:',gc.best_params_,gr.best_params_)
display(pd.DataFrame(rows)[pd.DataFrame(rows).lab.eq('ЛР1')])


Лучшие параметры: {'n_neighbors': 5} {'n_neighbors': 7}


,lab,task,model,implementation,stage,accuracy,precision_macro,recall_macro,f1_macro,MAE,RMSE,R2
0,ЛР1,classification,KNN без масштабирования,sklearn,baseline,0.983333,0.983449,0.983095,0.983046,NaN,NaN,NaN
1,ЛР1,regression,KNN без масштабирования,sklearn,baseline,NaN,NaN,NaN,NaN,42.648475,54.653225,0.436223
2,ЛР1,classification,KNN со стандартизацией,sklearn,baseline_scaled,0.963889,0.964590,0.963722,0.963440,NaN,NaN,NaN
3,ЛР1,regression,KNN со стандартизацией,sklearn,baseline_scaled,NaN,NaN,NaN,NaN,42.781701,54.493337,0.439517
4,ЛР1,classification,KNN настроенный,sklearn,tuned,0.983333,0.983449,0.983095,0.983046,NaN,NaN,NaN
5,ЛР1,regression,KNN настроенный,sklearn,tuned,NaN,NaN,NaN,NaN,42.648475,54.653225,0.436223
6,ЛР1,classification,KNN настроенный + стандартизация,sklearn,improved_sklearn,0.963889,0.964590,0.963722,0.963440,NaN,NaN,NaN
7,ЛР1,regression,KNN настроенный + стандартизация,sklearn,improved_sklearn,NaN,NaN,NaN,NaN,42.781701,54.493337,0.439517


## Пункт 4. Собственная реализация и вывод

`MyKNNClassifier` реализует поиск ближайших объектов и голосование по наиболее частому классу, `MyKNNRegressor` усредняет целевые значения соседей. В обоих классах масштабирование, если оно включено, рассчитывается только по обучающей выборке и затем применяется к тесту.




In [5]:
# Обучаем собственный KNN в тех же условиях и сравниваем его с библиотечной моделью.
add_cls(MyKNNClassifier(5,scale=False),rows,predictions,fitted,Xc_tr,yc_tr,Xc_te,yc_te,'MyKNN','ЛР1','собственная реализация','custom')
add_reg(MyKNNRegressor(7,scale=False),rows,predictions,fitted,Xr_tr,yr_tr,Xr_te,yr_te,'MyKNN','ЛР1','собственная реализация','custom')
add_cls(MyKNNClassifier(gc.best_params_['n_neighbors'],scale=True),rows,predictions,fitted,Xc_tr,yc_tr,Xc_te,yc_te,'MyKNN улучшенный','ЛР1','собственная реализация','custom_improved')
add_reg(MyKNNRegressor(gr.best_params_['n_neighbors'],scale=True),rows,predictions,fitted,Xr_tr,yr_tr,Xr_te,yr_te,'MyKNN улучшенный','ЛР1','собственная реализация','custom_improved')
display(pd.DataFrame(rows)[pd.DataFrame(rows).lab.eq('ЛР1')])


,lab,task,model,implementation,stage,accuracy,precision_macro,recall_macro,f1_macro,MAE,RMSE,R2
0,ЛР1,classification,KNN без масштабирования,sklearn,baseline,0.983333,0.983449,0.983095,0.983046,NaN,NaN,NaN
1,ЛР1,regression,KNN без масштабирования,sklearn,baseline,NaN,NaN,NaN,NaN,42.648475,54.653225,0.436223
2,ЛР1,classification,KNN со стандартизацией,sklearn,baseline_scaled,0.963889,0.964590,0.963722,0.963440,NaN,NaN,NaN
3,ЛР1,regression,KNN со стандартизацией,sklearn,baseline_scaled,NaN,NaN,NaN,NaN,42.781701,54.493337,0.439517
4,ЛР1,classification,KNN настроенный,sklearn,tuned,0.983333,0.983449,0.983095,0.983046,NaN,NaN,NaN
5,ЛР1,regression,KNN настроенный,sklearn,tuned,NaN,NaN,NaN,NaN,42.648475,54.653225,0.436223
6,ЛР1,classification,KNN настроенный + стандартизация,sklearn,improved_sklearn,0.963889,0.964590,0.963722,0.963440,NaN,NaN,NaN
7,ЛР1,regression,KNN настроенный + стандартизация,sklearn,improved_sklearn,NaN,NaN,NaN,NaN,42.781701,54.493337,0.439517
8,ЛР1,classification,MyKNN,собственная реализация,custom,0.983333,0.983449,0.983095,0.983046,NaN,NaN,NaN
9,ЛР1,regression,MyKNN,собственная реализация,custom,NaN,NaN,NaN,NaN,42.648475,54.653225,0.436223


**Вывод по ЛР1.** Подбор числа соседей не изменил оптимум относительно исходной конфигурации: CV выбрала `k=5` для классификации и `k=7` для регрессии — те же значения, с которых начинался baseline. Следовательно, сам по себе подбор `k` в данном эксперименте не дал выигрыша.

Для Digits стандартизация снизила macro-F1 с **0.9830 до 0.9634** (−0.0196), то есть гипотеза о безусловной пользе масштабирования для классификации не подтвердилась. Это согласуется с тем, что пиксельные признаки уже измеряются в общей шкале интенсивности. Для Diabetes RMSE немного уменьшился с **54.653 до 54.493** (−0.160), а R² вырос с **0.4362 до 0.4395** (+0.0033), но MAE слегка увеличился с 42.648 до 42.782. Значит, эффект стандартизации на регрессии небольшой и неоднозначный по метрикам.

Собственные реализации KNN воспроизводят результаты sklearn практически точно при одинаковых `k` и способе масштабирования. \
Главный результат работы — масштабирование не следует считать универсальным способом улучшения метрик.

# Лабораторная работа №2 — логистическая и линейная регрессия

## Пункт 1. Подготовка данных и метрики

Digits решается как многоклассовая классификация, Diabetes — как регрессия. Одинаковое разбиение с ЛР1 позволяет сравнивать результаты алгоритмов.

In [6]:
# Выполняем подготовку данных и показываем её результат.
print('Digits: обучение', Xc_tr.shape, 'тест', Xc_te.shape)
print('Diabetes: обучение', Xr_tr.shape, 'тест', Xr_te.shape)
print('Классов Digits:', yc.nunique(), 'признаков Digits:', Xc.shape[1], 'признаков Diabetes:', Xr.shape[1])


Digits: обучение (1437, 64) тест (360, 64)
Diabetes: обучение (353, 10) тест (89, 10)
Классов Digits: 10 признаков Digits: 64 признаков Diabetes: 10


## Пункт 2. Исходная модель и оценка

Используем LogisticRegression и LinearRegression из sklearn. В этой точке фиксируется качество без добавления новых признаков.

In [7]:
# Обучаем исходные библиотечные и собственные линейные модели.
add_cls(LogisticRegression(max_iter=2000),rows,predictions,fitted,Xc_tr,yc_tr,Xc_te,yc_te,'Логистическая регрессия','ЛР2','sklearn','baseline')
add_reg(LinearRegression(),rows,predictions,fitted,Xr_tr,yr_tr,Xr_te,yr_te,'Линейная регрессия','ЛР2','sklearn','baseline')
display(pd.DataFrame(rows)[pd.DataFrame(rows).lab.eq('ЛР2')])


,lab,task,model,implementation,stage,accuracy,precision_macro,recall_macro,f1_macro,MAE,RMSE,R2
12,ЛР2,classification,Логистическая регрессия,sklearn,baseline,0.958333,0.958731,0.958012,0.958075,NaN,NaN,NaN
13,ЛР2,regression,Линейная регрессия,sklearn,baseline,NaN,NaN,NaN,NaN,42.794095,53.853446,0.452603


## Пункт 3. Гипотеза и новые признаки

Гипотеза: агрегаты яркости изображения и медицинские взаимодействия добавят модели полезную информацию. Новые признаки формируются отдельно для каждого набора и проверяются на тестовой части.

In [8]:
# Формируем новые признаки, обучаем улучшенные модели и сравниваем их с исходными.
add_cls(Pipeline([('scale',StandardScaler()),('model',LogisticRegression(C=1,max_iter=2500))]),rows,predictions,fitted,Xcfe_tr,yc_tr,Xcfe_te,yc_te,'Логистическая регрессия + новые признаки','ЛР2','sklearn','feature_engineered')
add_reg(Pipeline([('scale',StandardScaler()),('model',Ridge(alpha=1))]),rows,predictions,fitted,Xrfe_tr,yr_tr,Xrfe_te,yr_te,'Ridge + новые признаки','ЛР2','sklearn','feature_engineered')
print('Добавлены признаки:',list(set(Xcfe.columns)-set(Xc.columns)),list(set(Xrfe.columns)-set(Xr.columns)))
display(pd.DataFrame(rows)[pd.DataFrame(rows).lab.eq('ЛР2')])


Добавлены признаки: ['pixel_mean', 'pixel_std', 'active_pixels'] ['s5_squared', 'bmi_squared', 'bmi_bp_interaction']


,lab,task,model,implementation,stage,accuracy,precision_macro,recall_macro,f1_macro,MAE,RMSE,R2
12,ЛР2,classification,Логистическая регрессия,sklearn,baseline,0.958333,0.958731,0.958012,0.958075,NaN,NaN,NaN
13,ЛР2,regression,Линейная регрессия,sklearn,baseline,NaN,NaN,NaN,NaN,42.794095,53.853446,0.452603
14,ЛР2,classification,Логистическая регрессия + новые признаки,sklearn,feature_engineered,0.980556,0.980839,0.980397,0.980430,NaN,NaN,NaN
15,ЛР2,regression,Ridge + новые признаки,sklearn,feature_engineered,NaN,NaN,NaN,NaN,42.105741,52.654356,0.476708


## Пункт 4. Собственная реализация и вывод

`MyLogisticRegression` — многоклассовая softmax-модель, обучаемая градиентным спуском с L2-регуляризацией и ранней остановкой. `MyLinearRegression` оценивает коэффициенты через псевдообратную матрицу. На следующем шаге к обеим собственным моделям применяются те же содержательные новые признаки, что и в эксперименте sklearn.




In [9]:
# Повторяем улучшение для собственных реализаций и выводим итоговую таблицу ЛР2.
add_cls(MyLogisticRegression(),rows,predictions,fitted,Xc_tr,yc_tr,Xc_te,yc_te,'Своя softmax-регрессия','ЛР2','собственная реализация','custom')
add_reg(MyLinearRegression(),rows,predictions,fitted,Xr_tr,yr_tr,Xr_te,yr_te,'Своя линейная регрессия','ЛР2','собственная реализация','custom')
add_cls(MyLogisticRegression(lr=.05,epochs=1800,reg=.001),rows,predictions,fitted,Xcfe_tr,yc_tr,Xcfe_te,yc_te,'Своя softmax-регрессия + новые признаки','ЛР2','собственная реализация','custom_improved')
add_reg(MyLinearRegression(),rows,predictions,fitted,Xrfe_tr,yr_tr,Xrfe_te,yr_te,'Своя линейная регрессия + новые признаки','ЛР2','собственная реализация','custom_improved')
display(pd.DataFrame(rows)[pd.DataFrame(rows).lab.eq('ЛР2')])


,lab,task,model,implementation,stage,accuracy,precision_macro,recall_macro,f1_macro,MAE,RMSE,R2
12,ЛР2,classification,Логистическая регрессия,sklearn,baseline,0.958333,0.958731,0.958012,0.958075,NaN,NaN,NaN
13,ЛР2,regression,Линейная регрессия,sklearn,baseline,NaN,NaN,NaN,NaN,42.794095,53.853446,0.452603
14,ЛР2,classification,Логистическая регрессия + новые признаки,sklearn,feature_engineered,0.980556,0.980839,0.980397,0.980430,NaN,NaN,NaN
15,ЛР2,regression,Ridge + новые признаки,sklearn,feature_engineered,NaN,NaN,NaN,NaN,42.105741,52.654356,0.476708
16,ЛР2,classification,Своя softmax-регрессия,собственная реализация,custom,0.958333,0.959047,0.958087,0.958137,NaN,NaN,NaN
17,ЛР2,regression,Своя линейная регрессия,собственная реализация,custom,NaN,NaN,NaN,NaN,42.794095,53.853446,0.452603
18,ЛР2,classification,Своя softmax-регрессия + новые признаки,собственная реализация,custom_improved,0.975000,0.975842,0.974916,0.975020,NaN,NaN,NaN
19,ЛР2,regression,Своя линейная регрессия + новые признаки,собственная реализация,custom_improved,NaN,NaN,NaN,NaN,41.170222,51.891580,0.491759


**Вывод по ЛР2.** Добавление трёх агрегатов яркости для Digits повысило macro-F1 sklearn с **0.9581 до 0.9804** (+0.0224), а Accuracy — с 0.9583 до 0.9806. Для Diabetes добавление взаимодействия BMI×BP и квадратов BMI и S5 снизило RMSE с **53.853 до 52.654** (−1.199), MAE — с 42.794 до 42.106 и повысило R² с 0.4526 до 0.4767. Поэтому гипотеза о полезности инженерии признаков поддержана на данном разбиении в обеих задачах.

Собственная softmax-модель с новыми признаками получила macro-F1 **0.9750**: это заметно выше её baseline (0.9581), но на 0.0054 ниже sklearn-модели с теми же новыми признаками (0.9804). Собственная линейная регрессия с новыми признаками достигла RMSE **51.892** и R² **0.4918**, то есть на этом разбиении превзошла Ridge с новыми признаками (RMSE 52.654; R² 0.4767). Различие объясняется не только алгоритмом: собственная модель не использует L2-штраф, тогда как Ridge его использует.

# Лабораторная работа №3 — решающее дерево

## Пункт 1. Подготовка данных и метрики

Для классификации используем Digits, для регрессии — Diabetes. Разбиение и метрики остаются теми же, что в предыдущих работах.

In [10]:
# Показываем размеры обучающей и тестовой частей перед обучением.
print('Digits: обучение', Xc_tr.shape, 'тест', Xc_te.shape)
print('Diabetes: обучение', Xr_tr.shape, 'тест', Xr_te.shape)
print('Классов Digits:', yc.nunique(), 'признаков Digits:', Xc.shape[1], 'признаков Diabetes:', Xr.shape[1])


Digits: обучение (1437, 64) тест (360, 64)
Diabetes: обучение (353, 10) тест (89, 10)
Классов Digits: 10 признаков Digits: 64 признаков Diabetes: 10


## Пункт 2. Исходная модель и оценка

Сначала обучаем библиотечную модель без поиска параметров. Это исходная точка сравнения.

In [11]:
# Обучаем исходные модели решающее дерево и выводим их метрики.
add_cls(DecisionTreeClassifier(random_state=RANDOM_STATE),rows,predictions,fitted,Xc_tr,yc_tr,Xc_te,yc_te,'ЛР3 решающее дерево','ЛР3','sklearn','baseline')
add_reg(DecisionTreeRegressor(random_state=RANDOM_STATE),rows,predictions,fitted,Xr_tr,yr_tr,Xr_te,yr_te,'ЛР3 решающее дерево','ЛР3','sklearn','baseline')
display(pd.DataFrame(rows)[pd.DataFrame(rows).lab.eq('ЛР3')])


,lab,task,model,implementation,stage,accuracy,precision_macro,recall_macro,f1_macro,MAE,RMSE,R2
20,ЛР3,classification,ЛР3 решающее дерево,sklearn,baseline,0.825,0.823584,0.824247,0.823049,NaN,NaN,NaN
21,ЛР3,regression,ЛР3 решающее дерево,sklearn,baseline,NaN,NaN,NaN,NaN,54.52809,70.546423,0.060654


## Пункт 3. Гипотеза и подбор параметров

Гипотеза: подбор глубины/размера листа (для дерева), числа деревьев и ограничений сложности (для леса), а также числа итераций, скорости обучения и глубины (для бустинга) повысит качество на отложенных данных. Параметры выбираются 5-fold CV только на обучающей части; тестовая выборка не участвует в подборе.

In [12]:
# Подбираем параметры решающее дерево и выводим сравнение с исходной моделью.
cv_c=StratifiedKFold(5,shuffle=True,random_state=RANDOM_STATE); cv_r=KFold(5,shuffle=True,random_state=RANDOM_STATE)
gc=GridSearchCV(DecisionTreeClassifier(random_state=RANDOM_STATE),{'max_depth':[5,10,15,None],'min_samples_leaf':[1,2,5]},cv=cv_c,scoring='f1_macro').fit(Xc_tr,yc_tr)
gr=GridSearchCV(DecisionTreeRegressor(random_state=RANDOM_STATE),{'max_depth':[3,5,8,None],'min_samples_leaf':[1,3,7]},cv=cv_r,scoring='neg_root_mean_squared_error').fit(Xr_tr,yr_tr)
add_cls(gc.best_estimator_,rows,predictions,fitted,Xc_tr,yc_tr,Xc_te,yc_te,'ЛР3 решающее дерево настроенный','ЛР3','sklearn','tuned')
add_reg(gr.best_estimator_,rows,predictions,fitted,Xr_tr,yr_tr,Xr_te,yr_te,'ЛР3 решающее дерево настроенный','ЛР3','sklearn','tuned')
best_params['ЛР3'] = {'classification': gc.best_params_, 'regression': gr.best_params_}
print('Лучшие параметры:',gc.best_params_,gr.best_params_)
display(pd.DataFrame(rows)[pd.DataFrame(rows).lab.eq('ЛР3')])


Лучшие параметры: {'max_depth': 15, 'min_samples_leaf': 1} {'max_depth': 3, 'min_samples_leaf': 1}


,lab,task,model,implementation,stage,accuracy,precision_macro,recall_macro,f1_macro,MAE,RMSE,R2
20,ЛР3,classification,ЛР3 решающее дерево,sklearn,baseline,0.825,0.823584,0.824247,0.823049,NaN,NaN,NaN
21,ЛР3,regression,ЛР3 решающее дерево,sklearn,baseline,NaN,NaN,NaN,NaN,54.528090,70.546423,0.060654
22,ЛР3,classification,ЛР3 решающее дерево настроенный,sklearn,tuned,0.825,0.823584,0.824247,0.823049,NaN,NaN,NaN
23,ЛР3,regression,ЛР3 решающее дерево настроенный,sklearn,tuned,NaN,NaN,NaN,NaN,48.096592,59.604541,0.329445


## Пункт 4. Собственная реализация и вывод

Собственное дерево реализует CART-разбиения: для классификации оценивается уменьшение неоднородности Джини, для регрессии — уменьшение суммы квадратов ошибок. 



В следующей кодовой ячейке собственная модель повторно обучается с техникой улучшения из пункта 3 (параметры CV), получает стадию `custom_improved` и сравнивается с tuned sklearn-моделью. Результаты и разности метрик сохраняются в итоговые CSV.


In [13]:
# Обучаем собственное дерево baseline и переносим подобранные параметры в custom_improved.
add_cls(MyDecisionTree('classification',max_depth=10),rows,predictions,fitted,Xc_tr,yc_tr,Xc_te,yc_te,'Своя модель решающее дерево','ЛР3','собственная реализация','custom')
add_reg(MyDecisionTree('regression',max_depth=5),rows,predictions,fitted,Xr_tr,yr_tr,Xr_te,yr_te,'Своя модель решающее дерево','ЛР3','собственная реализация','custom')
add_cls(MyDecisionTree('classification',max_depth=gc.best_params_['max_depth'],min_samples_leaf=gc.best_params_['min_samples_leaf']),rows,predictions,fitted,Xc_tr,yc_tr,Xc_te,yc_te,'Своя модель дерево + параметры CV','ЛР3','собственная реализация','custom_improved')
add_reg(MyDecisionTree('regression',max_depth=gr.best_params_['max_depth'],min_samples_leaf=gr.best_params_['min_samples_leaf']),rows,predictions,fitted,Xr_tr,yr_tr,Xr_te,yr_te,'Своя модель дерево + параметры CV','ЛР3','собственная реализация','custom_improved')
display(pd.DataFrame(rows)[pd.DataFrame(rows).lab.eq('ЛР3')])


,lab,task,model,implementation,stage,accuracy,precision_macro,recall_macro,f1_macro,MAE,RMSE,R2
20,ЛР3,classification,ЛР3 решающее дерево,sklearn,baseline,0.825000,0.823584,0.824247,0.823049,NaN,NaN,NaN
21,ЛР3,regression,ЛР3 решающее дерево,sklearn,baseline,NaN,NaN,NaN,NaN,54.528090,70.546423,0.060654
22,ЛР3,classification,ЛР3 решающее дерево настроенный,sklearn,tuned,0.825000,0.823584,0.824247,0.823049,NaN,NaN,NaN
23,ЛР3,regression,ЛР3 решающее дерево настроенный,sklearn,tuned,NaN,NaN,NaN,NaN,48.096592,59.604541,0.329445
24,ЛР3,classification,Своя модель решающее дерево,собственная реализация,custom,0.844444,0.848408,0.844314,0.841860,NaN,NaN,NaN
25,ЛР3,regression,Своя модель решающее дерево,собственная реализация,custom,NaN,NaN,NaN,NaN,47.267142,59.069458,0.341431
26,ЛР3,classification,Своя модель дерево + параметры CV,собственная реализация,custom_improved,0.763889,0.765827,0.763419,0.762739,NaN,NaN,NaN
27,ЛР3,regression,Своя модель дерево + параметры CV,собственная реализация,custom_improved,NaN,NaN,NaN,NaN,47.793953,60.774869,0.302854


**Вывод по ЛР3.** Для классификации GridSearchCV выбрал глубину 15 и `min_samples_leaf=1`, но macro-F1 остался **0.8230** — ровно на уровне baseline. Следовательно, гипотеза об улучшении классификации настройкой дерева не подтвердилась на данном тестовом разбиении. Для регрессии настройка с глубиной 3 снизила RMSE с **70.546 до 59.605** (−10.942, около 15.5%), MAE — с 54.528 до 48.097, а R² вырос с 0.0607 до 0.3294. Для регрессии контроль глубины существенно уменьшил ошибку исходного дерева.

Исходная собственная реализация получила macro-F1 **0.8419**, немного выше sklearn-дерева (0.8230), а её RMSE регрессии равен **59.069** — немного лучше tuned sklearn (59.605). После переноса ограничений сложности в `custom_improved` собственные показатели стали хуже: macro-F1 снизился до **0.7627**, RMSE вырос до **60.775**. Это важный отрицательный результат: параметры, выбранные для sklearn-CART, не обязаны переноситься на дерево с иным механизмом отбора признаков и порогов. 

# Лабораторная работа №4 — случайный лес

## Пункт 1. Подготовка данных и метрики

Для классификации используем Digits, для регрессии — Diabetes. Разбиение и метрики остаются теми же, что в предыдущих работах.

In [14]:
# Показываем размеры обучающей и тестовой частей перед обучением.
print('Digits: обучение', Xc_tr.shape, 'тест', Xc_te.shape)
print('Diabetes: обучение', Xr_tr.shape, 'тест', Xr_te.shape)
print('Классов Digits:', yc.nunique(), 'признаков Digits:', Xc.shape[1], 'признаков Diabetes:', Xr.shape[1])


Digits: обучение (1437, 64) тест (360, 64)
Diabetes: обучение (353, 10) тест (89, 10)
Классов Digits: 10 признаков Digits: 64 признаков Diabetes: 10


## Пункт 2. Исходная модель и оценка

Сначала обучаем библиотечную модель без поиска параметров. Это исходная точка сравнения.

In [15]:
# Обучаем исходные модели случайный лес и выводим их метрики.
add_cls(RandomForestClassifier(random_state=RANDOM_STATE,n_jobs=1),rows,predictions,fitted,Xc_tr,yc_tr,Xc_te,yc_te,'ЛР4 случайный лес','ЛР4','sklearn','baseline')
add_reg(RandomForestRegressor(random_state=RANDOM_STATE,n_jobs=1),rows,predictions,fitted,Xr_tr,yr_tr,Xr_te,yr_te,'ЛР4 случайный лес','ЛР4','sklearn','baseline')
display(pd.DataFrame(rows)[pd.DataFrame(rows).lab.eq('ЛР4')])


,lab,task,model,implementation,stage,accuracy,precision_macro,recall_macro,f1_macro,MAE,RMSE,R2
28,ЛР4,classification,ЛР4 случайный лес,sklearn,baseline,0.961111,0.96197,0.96071,0.960722,NaN,NaN,NaN
29,ЛР4,regression,ЛР4 случайный лес,sklearn,baseline,NaN,NaN,NaN,NaN,44.053034,54.332408,0.442823


## Пункт 3. Гипотеза и подбор параметров

Гипотеза: подбор глубины/размера листа (для дерева), числа деревьев и ограничений сложности (для леса), а также числа итераций, скорости обучения и глубины (для бустинга) повысит качество на отложенных данных. Параметры выбираются 5-fold CV только на обучающей части; тестовая выборка не участвует в подборе.

In [16]:
# Подбираем параметры случайный лес и выводим сравнение с исходной моделью.
cv_c=StratifiedKFold(5,shuffle=True,random_state=RANDOM_STATE); cv_r=KFold(5,shuffle=True,random_state=RANDOM_STATE)
gc=GridSearchCV(RandomForestClassifier(random_state=RANDOM_STATE,n_jobs=1),{'n_estimators':[100,150],'max_depth':[None,15],'min_samples_leaf':[1]},cv=cv_c,scoring='f1_macro').fit(Xc_tr,yc_tr)
gr=GridSearchCV(RandomForestRegressor(random_state=RANDOM_STATE,n_jobs=1),{'n_estimators':[100,150],'max_depth':[8,15],'min_samples_leaf':[1]},cv=cv_r,scoring='neg_root_mean_squared_error').fit(Xr_tr,yr_tr)
add_cls(gc.best_estimator_,rows,predictions,fitted,Xc_tr,yc_tr,Xc_te,yc_te,'ЛР4 случайный лес настроенный','ЛР4','sklearn','tuned')
add_reg(gr.best_estimator_,rows,predictions,fitted,Xr_tr,yr_tr,Xr_te,yr_te,'ЛР4 случайный лес настроенный','ЛР4','sklearn','tuned')
best_params['ЛР4'] = {'classification': gc.best_params_, 'regression': gr.best_params_}
print('Лучшие параметры:',gc.best_params_,gr.best_params_)
display(pd.DataFrame(rows)[pd.DataFrame(rows).lab.eq('ЛР4')])


Лучшие параметры: {'max_depth': None, 'min_samples_leaf': 1, 'n_estimators': 150} {'max_depth': 8, 'min_samples_leaf': 1, 'n_estimators': 150}


,lab,task,model,implementation,stage,accuracy,precision_macro,recall_macro,f1_macro,MAE,RMSE,R2
28,ЛР4,classification,ЛР4 случайный лес,sklearn,baseline,0.961111,0.96197,0.96071,0.960722,NaN,NaN,NaN
29,ЛР4,regression,ЛР4 случайный лес,sklearn,baseline,NaN,NaN,NaN,NaN,44.053034,54.332408,0.442823
30,ЛР4,classification,ЛР4 случайный лес настроенный,sklearn,tuned,0.961111,0.96197,0.96071,0.960722,NaN,NaN,NaN
31,ЛР4,regression,ЛР4 случайный лес настроенный,sklearn,tuned,NaN,NaN,NaN,NaN,44.231746,54.490419,0.439577


## Пункт 4. Собственная реализация и вывод

Собственный случайный лес обучает CART-деревья на bootstrap-выборках, а затем усредняет регрессионные прогнозы или голосует за класс. Для улучшенной собственной версии число деревьев увеличивается до 30; глубина ограничивается восемью уровнями, а минимальный размер листа — не менее двух объектов. 






In [17]:
# Обучаем собственный bootstrap-лес baseline и переносим параметры CV в custom_improved.
add_cls(MyRandomForest('classification'),rows,predictions,fitted,Xc_tr,yc_tr,Xc_te,yc_te,'Своя модель случайный лес','ЛР4','собственная реализация','custom')
add_reg(MyRandomForest('regression'),rows,predictions,fitted,Xr_tr,yr_tr,Xr_te,yr_te,'Своя модель случайный лес','ЛР4','собственная реализация','custom')
forest_cls_params={**gc.best_params_,'n_estimators':min(gc.best_params_['n_estimators'],30),'max_depth':min(gc.best_params_['max_depth'] or 8,8),'min_samples_leaf':max(gc.best_params_['min_samples_leaf'],2)}
forest_reg_params={**gr.best_params_,'n_estimators':min(gr.best_params_['n_estimators'],30),'max_depth':min(gr.best_params_['max_depth'] or 8,8),'min_samples_leaf':max(gr.best_params_['min_samples_leaf'],2)}
add_cls(MyRandomForest('classification',**forest_cls_params),rows,predictions,fitted,Xc_tr,yc_tr,Xc_te,yc_te,'Своя модель лес + параметры CV','ЛР4','собственная реализация','custom_improved')
add_reg(MyRandomForest('regression',**forest_reg_params),rows,predictions,fitted,Xr_tr,yr_tr,Xr_te,yr_te,'Своя модель лес + параметры CV','ЛР4','собственная реализация','custom_improved')
display(pd.DataFrame(rows)[pd.DataFrame(rows).lab.eq('ЛР4')])


,lab,task,model,implementation,stage,accuracy,precision_macro,recall_macro,f1_macro,MAE,RMSE,R2
28,ЛР4,classification,ЛР4 случайный лес,sklearn,baseline,0.961111,0.961970,0.960710,0.960722,NaN,NaN,NaN
29,ЛР4,regression,ЛР4 случайный лес,sklearn,baseline,NaN,NaN,NaN,NaN,44.053034,54.332408,0.442823
30,ЛР4,classification,ЛР4 случайный лес настроенный,sklearn,tuned,0.961111,0.961970,0.960710,0.960722,NaN,NaN,NaN
31,ЛР4,regression,ЛР4 случайный лес настроенный,sklearn,tuned,NaN,NaN,NaN,NaN,44.231746,54.490419,0.439577
32,ЛР4,classification,Своя модель случайный лес,собственная реализация,custom,0.863889,0.865138,0.863604,0.862041,NaN,NaN,NaN
33,ЛР4,regression,Своя модель случайный лес,собственная реализация,custom,NaN,NaN,NaN,NaN,42.494447,53.088426,0.468045
34,ЛР4,classification,Своя модель лес + параметры CV,собственная реализация,custom_improved,0.958333,0.960252,0.958091,0.958152,NaN,NaN,NaN
35,ЛР4,regression,Своя модель лес + параметры CV,собственная реализация,custom_improved,NaN,NaN,NaN,NaN,43.947628,54.878157,0.431573


**Вывод по ЛР4.** В текущей сетке GridSearchCV для Digits выбрал лес из 150 деревьев без ограничения глубины, но macro-F1 остался **0.9607** — на уровне baseline. Для Diabetes выбран лес из 150 деревьев глубиной до 8; RMSE изменился с **54.332 до 54.490** (+0.158), а R² снизился с 0.4428 до 0.4396. То есть настройка не улучшила ни одну целевую метрику на этом holdout-разбиении; гипотеза H3 для ЛР4 не подтверждена.

При этом сравнение tuned-леса с tuned-деревом из ЛР3 поддерживает гипотезу об ансамблировании: macro-F1 выше (**0.9607 против 0.8230**, +0.1377), а RMSE ниже (**54.490 против 59.605**, −5.114). Исходная собственная версия леса имела macro-F1 **0.8620** и RMSE **53.088**. После увеличения ансамбля до 30 деревьев и переноса ограничений сложности macro-F1 вырос до **0.9582** (разница с tuned sklearn — 0.0026), но RMSE стал **54.878**, то есть хуже и собственной исходной версии, и tuned sklearn (54.490). Это показывает, что увеличение bootstrap-ансамбля заметно помогло классификации, но не гарантировало аналогичного эффекта в регрессии. 

**Практический вывод:** добавление ансамблирования в этой работе дало более заметное отличие от одиночного дерева, чем локальная настройка параметров самого леса. Для Digits преимущество леса по сравнению с деревом выражено сильнее; для Diabetes абсолютная ошибка леса тоже ниже, но качество зависит от выбранной глубины и состава bootstrap-выборок.

# Лабораторная работа №5 — градиентный бустинг

## Пункт 1. Подготовка данных и метрики

Для классификации используем Digits, для регрессии — Diabetes. Разбиение и метрики остаются теми же, что в предыдущих работах.

In [18]:
# Показываем размеры обучающей и тестовой частей перед обучением.
print('Digits: обучение', Xc_tr.shape, 'тест', Xc_te.shape)
print('Diabetes: обучение', Xr_tr.shape, 'тест', Xr_te.shape)
print('Классов Digits:', yc.nunique(), 'признаков Digits:', Xc.shape[1], 'признаков Diabetes:', Xr.shape[1])


Digits: обучение (1437, 64) тест (360, 64)
Diabetes: обучение (353, 10) тест (89, 10)
Классов Digits: 10 признаков Digits: 64 признаков Diabetes: 10


## Пункт 2. Исходная модель и оценка

Сначала обучаем библиотечную гистограммную реализацию градиентного бустинга без поиска параметров. Это исходная точка сравнения; гистограммный вариант позволяет ускорить CV на многоклассовом Digits, сохраняя принцип последовательного построения деревьев.

In [19]:
# Обучаем исходные модели градиентный бустинг и выводим их метрики.
add_cls(HistGradientBoostingClassifier(random_state=RANDOM_STATE,min_samples_leaf=10),rows,predictions,fitted,Xc_tr,yc_tr,Xc_te,yc_te,'ЛР5 гистограммный градиентный бустинг','ЛР5','sklearn','baseline')
add_reg(HistGradientBoostingRegressor(random_state=RANDOM_STATE,min_samples_leaf=5),rows,predictions,fitted,Xr_tr,yr_tr,Xr_te,yr_te,'ЛР5 гистограммный градиентный бустинг','ЛР5','sklearn','baseline')
display(pd.DataFrame(rows)[pd.DataFrame(rows).lab.eq('ЛР5')])


,lab,task,model,implementation,stage,accuracy,precision_macro,recall_macro,f1_macro,MAE,RMSE,R2
36,ЛР5,classification,ЛР5 гистограммный градиентный бустинг,sklearn,baseline,0.955556,0.955918,0.955,0.954733,NaN,NaN,NaN
37,ЛР5,regression,ЛР5 гистограммный градиентный бустинг,sklearn,baseline,NaN,NaN,NaN,NaN,45.204033,57.029245,0.386138


## Пункт 3. Гипотеза и подбор параметров

Гипотеза: подбор глубины/размера листа (для дерева), числа деревьев и ограничений сложности (для леса), а также числа итераций, скорости обучения и числа листьев базовых деревьев (для бустинга) повысит качество на отложенных данных. Параметры выбираются 5-fold CV только на обучающей части; тестовая выборка не участвует в подборе.

In [20]:
# Подбираем параметры гистограммного градиентного бустинга и сравниваем его с исходной моделью.
cv_c=StratifiedKFold(5,shuffle=True,random_state=RANDOM_STATE); cv_r=KFold(5,shuffle=True,random_state=RANDOM_STATE)
gc=GridSearchCV(HistGradientBoostingClassifier(random_state=RANDOM_STATE,min_samples_leaf=10),[
    {'max_iter':[50],'learning_rate':[.1],'max_leaf_nodes':[15]},
    {'max_iter':[100],'learning_rate':[.05],'max_leaf_nodes':[7]},
],cv=cv_c,scoring='f1_macro').fit(Xc_tr,yc_tr)
gr=GridSearchCV(HistGradientBoostingRegressor(random_state=RANDOM_STATE,min_samples_leaf=5),[
    {'max_iter':[50],'learning_rate':[.05],'max_leaf_nodes':[7]},
    {'max_iter':[100],'learning_rate':[.03],'max_leaf_nodes':[15]},
],cv=cv_r,scoring='neg_root_mean_squared_error').fit(Xr_tr,yr_tr)
add_cls(gc.best_estimator_,rows,predictions,fitted,Xc_tr,yc_tr,Xc_te,yc_te,'ЛР5 гистограммный градиентный бустинг настроенный','ЛР5','sklearn','tuned')
add_reg(gr.best_estimator_,rows,predictions,fitted,Xr_tr,yr_tr,Xr_te,yr_te,'ЛР5 гистограммный градиентный бустинг настроенный','ЛР5','sklearn','tuned')
best_params['ЛР5'] = {'classification': gc.best_params_, 'regression': gr.best_params_}
print('Лучшие параметры:',gc.best_params_,gr.best_params_)
display(pd.DataFrame(rows)[pd.DataFrame(rows).lab.eq('ЛР5')])


Лучшие параметры: {'learning_rate': 0.1, 'max_iter': 50, 'max_leaf_nodes': 15} {'learning_rate': 0.05, 'max_iter': 50, 'max_leaf_nodes': 7}


,lab,task,model,implementation,stage,accuracy,precision_macro,recall_macro,f1_macro,MAE,RMSE,R2
36,ЛР5,classification,ЛР5 гистограммный градиентный бустинг,sklearn,baseline,0.955556,0.955918,0.955000,0.954733,NaN,NaN,NaN
37,ЛР5,regression,ЛР5 гистограммный градиентный бустинг,sklearn,baseline,NaN,NaN,NaN,NaN,45.204033,57.029245,0.386138
38,ЛР5,classification,ЛР5 гистограммный градиентный бустинг настроенный,sklearn,tuned,0.952778,0.953591,0.952222,0.952240,NaN,NaN,NaN
39,ЛР5,regression,ЛР5 гистограммный градиентный бустинг настроенный,sklearn,tuned,NaN,NaN,NaN,NaN,43.056510,53.062205,0.468570


## Пункт 4. Собственная реализация и вывод

Собственный градиентный бустинг строит деревья последовательно. В регрессии очередное дерево приближает остатки между истинными и текущими прогнозами (градиент квадратичной функции потерь). В классификации используется многоклассовая схема: на каждом шаге регрессионные деревья приближают отрицательный градиент softmax-cross-entropy для каждого класса, после чего обновляются оценки классов.

Для sklearn используется **HistGradientBoostingClassifier/Regressor** — гистограммная разновидность градиентного бустинга. Она сохраняет последовательный принцип построения деревьев и позволяет провести 5-fold CV на Digits в разумное время. В CV проверяются два конфигурационных варианта; собственная версия сохраняет 15 итераций, использует выбранную скорость обучения и приближённую глубину (до трёх уровней).



Собственная модель использует подобранную скорость обучения и приближённую глубину; число итераций учебной реализации оставлено равным 15, чтобы запуск оставался воспроизводимым по времени. Результат получает стадию `custom_improved` и сравнивается с tuned sklearn-моделью.


In [21]:
# Обучаем собственный градиентный бустинг для классификации в исходной конфигурации.
add_cls(MyGradientBoosting('classification'),rows,predictions,fitted,Xc_tr,yc_tr,Xc_te,yc_te,'Своя модель градиентный бустинг','ЛР5','собственная реализация','custom')


In [22]:
# Обучаем собственный градиентный бустинг для регрессии в исходной конфигурации.
add_reg(MyGradientBoosting('regression'),rows,predictions,fitted,Xr_tr,yr_tr,Xr_te,yr_te,'Своя модель градиентный бустинг','ЛР5','собственная реализация','custom')


In [23]:
# Применяем CV-скорость обучения и приближённую глубину к собственной модели классификации.
boost_cls_params={'n_estimators':15,'learning_rate':gc.best_params_['learning_rate'],'max_depth':min(int(np.ceil(np.log2(gc.best_params_['max_leaf_nodes']))),3)}
add_cls(MyGradientBoosting('classification',**boost_cls_params),rows,predictions,fitted,Xc_tr,yc_tr,Xc_te,yc_te,'Своя модель бустинг + параметры CV','ЛР5','собственная реализация','custom_improved')


In [24]:
# Применяем CV-скорость обучения и приближённую глубину к собственной модели регрессии.
boost_reg_params={'n_estimators':15,'learning_rate':gr.best_params_['learning_rate'],'max_depth':min(int(np.ceil(np.log2(gr.best_params_['max_leaf_nodes']))),3)}
add_reg(MyGradientBoosting('regression',**boost_reg_params),rows,predictions,fitted,Xr_tr,yr_tr,Xr_te,yr_te,'Своя модель бустинг + параметры CV','ЛР5','собственная реализация','custom_improved')


In [25]:
# Сводим метрики двух собственных вариантов градиентного бустинга.
display(pd.DataFrame(rows)[pd.DataFrame(rows).lab.eq('ЛР5')])


,lab,task,model,implementation,stage,accuracy,precision_macro,recall_macro,f1_macro,MAE,RMSE,R2
36,ЛР5,classification,ЛР5 гистограммный градиентный бустинг,sklearn,baseline,0.955556,0.955918,0.955000,0.954733,NaN,NaN,NaN
37,ЛР5,regression,ЛР5 гистограммный градиентный бустинг,sklearn,baseline,NaN,NaN,NaN,NaN,45.204033,57.029245,0.386138
38,ЛР5,classification,ЛР5 гистограммный градиентный бустинг настроенный,sklearn,tuned,0.952778,0.953591,0.952222,0.952240,NaN,NaN,NaN
39,ЛР5,regression,ЛР5 гистограммный градиентный бустинг настроенный,sklearn,tuned,NaN,NaN,NaN,NaN,43.056510,53.062205,0.468570
40,ЛР5,classification,Своя модель градиентный бустинг,собственная реализация,custom,0.827778,0.831849,0.827104,0.825678,NaN,NaN,NaN
41,ЛР5,regression,Своя модель градиентный бустинг,собственная реализация,custom,NaN,NaN,NaN,NaN,46.170385,55.265639,0.423518
42,ЛР5,classification,Своя модель бустинг + параметры CV,собственная реализация,custom_improved,0.875000,0.879888,0.874260,0.873458,NaN,NaN,NaN
43,ЛР5,regression,Своя модель бустинг + параметры CV,собственная реализация,custom_improved,NaN,NaN,NaN,NaN,51.448420,59.398769,0.334067


**Вывод по ЛР5.** В классификации tuned HistGradientBoosting получил macro-F1 **0.9522**, что ниже baseline **0.9547** на 0.0025; гипотеза об улучшении классификации настройкой не подтвердилась. В регрессии настройка дала выраженный выигрыш: RMSE уменьшился с **57.029 до 53.062** (−3.967, около 7.0%), MAE — с 45.204 до 43.057, а R² вырос с 0.3861 до 0.4686 (+0.0824). Таким образом, H3 для ЛР5 поддержана в регрессии, но опровергнута в классификации.

Собственная модель классификации улучшилась после переноса скорости обучения и глубины: macro-F1 вырос с **0.8257 до 0.8735**, но остался ниже tuned sklearn (0.9522). В регрессии собственный baseline имел RMSE **55.266** и был лучше исходного HistGradientBoosting (57.029), но после переноса параметров RMSE вырос до **59.399**, то есть качество ухудшилось. Это важный отрицательный результат: параметры гистограммного sklearn-бустинга не переносятся один к одному на собственную реализацию CART, а приближение числа листьев глубиной дерева меняет поведение модели. 

**Практический вывод:** градиентный бустинг полезен для прогноза Diabetes в этой постановке, но даже после CV качество зависит от числа итераций, learning rate и сложности базовых деревьев. Для Digits на данном split он уступил KNN и случайному лесу.

# Общий итоговый вывод

В пяти лабораторных работах сопоставлены KNN, логистическая и линейная регрессии, дерево решений, случайный лес и градиентный бустинг на одинаковых разбиениях Digits и `load_diabetes`. В классификации лучший результат среди всех проверенных sklearn-вариантов дал KNN без стандартизации: **macro-F1 = 0.9830**. Стандартизованный KNN имел macro-F1 0.9634, поэтому масштабирование в этой задаче ухудшило результат. Для регрессии лучший sklearn-вариант — Ridge с новыми признаками из ЛР2 (**RMSE = 52.654, R² = 0.4767**); среди собственных реализаций лучше всего сработала линейная регрессия с новыми признаками (**RMSE = 51.892, R² = 0.4918**).

Проверка гипотез дала неоднозначную картину. **H1 (масштабирование KNN):** для классификации не подтвердилась (macro-F1 снизился на 0.0196), для регрессии поддержана по RMSE (снижение на 0.160) и R² (+0.0033), хотя MAE немного вырос. **H2 (новые признаки):** поддержана для обеих задач: macro-F1 Digits вырос с 0.9581 до 0.9804, RMSE Diabetes снизился с 53.853 до 52.654, R² вырос с 0.4526 до 0.4767. **H3 (настройка):** в ЛР3 улучшилась только регрессия (RMSE −10.942); в ЛР4 tuning не улучшил ни одну целевую метрику; в ЛР5 регрессия улучшилась (RMSE −3.967), а классификация слегка ухудшилась (macro-F1 −0.0025). Следовательно, подбор параметров нельзя автоматически трактовать как улучшение.

**H4 (ансамбли против одиночного дерева)** поддержана сравнением tuned-моделей ЛР3 и ЛР4: у леса macro-F1 выше на 0.1377, а RMSE ниже на 5.114. Это показывает преимущество ансамбля на текущем разбиении.

Собственные алгоритмы реализуют основные механизмы методов и дают смешанные результаты. MyKNN совпадает со sklearn при совпадающих настройках. MyLinearRegression с новыми признаками дала лучший RMSE среди проверенных регрессоров, собственные дерево и лес приблизились к отдельным sklearn-показателям, а собственный бустинг при переносе параметров показал ухудшение на регрессии. Ограничения учебных реализаций — ограниченный поиск признаков/порогов CART, небольшое число деревьев/итераций и приближённый перенос параметров — объясняют расхождения.

Важные ограничения: результаты рассчитаны на одном фиксированном разбиении с `random_state=42`.


In [26]:
# Показываем общий рейтинг reference-моделей и основные результаты по двум задачам.
reference_stages = ['baseline', 'tuned', 'feature_engineered', 'improved_sklearn', 'custom', 'custom_improved']
reference_models = pd.DataFrame(rows)
reference_models = reference_models[reference_models['stage'].isin(reference_stages)]
print('Лучшие reference-модели классификации по macro-F1:')
display(reference_models[reference_models['task']=='classification'].sort_values('f1_macro',ascending=False).head(10))
print('Лучшие reference-модели регрессии по RMSE:')
display(reference_models[reference_models['task']=='regression'].sort_values('RMSE').head(10))


Лучшие reference-модели классификации по macro-F1:


,lab,task,model,implementation,stage,accuracy,precision_macro,recall_macro,f1_macro,MAE,RMSE,R2
0,ЛР1,classification,KNN без масштабирования,sklearn,baseline,0.983333,0.983449,0.983095,0.983046,NaN,NaN,NaN
4,ЛР1,classification,KNN настроенный,sklearn,tuned,0.983333,0.983449,0.983095,0.983046,NaN,NaN,NaN
8,ЛР1,classification,MyKNN,собственная реализация,custom,0.983333,0.983449,0.983095,0.983046,NaN,NaN,NaN
14,ЛР2,classification,Логистическая регрессия + новые признаки,sklearn,feature_engineered,0.980556,0.980839,0.980397,0.980430,NaN,NaN,NaN
18,ЛР2,classification,Своя softmax-регрессия + новые признаки,собственная реализация,custom_improved,0.975000,0.975842,0.974916,0.975020,NaN,NaN,NaN
10,ЛР1,classification,MyKNN улучшенный,собственная реализация,custom_improved,0.963889,0.964590,0.963722,0.963440,NaN,NaN,NaN
6,ЛР1,classification,KNN настроенный + стандартизация,sklearn,improved_sklearn,0.963889,0.964590,0.963722,0.963440,NaN,NaN,NaN
30,ЛР4,classification,ЛР4 случайный лес настроенный,sklearn,tuned,0.961111,0.961970,0.960710,0.960722,NaN,NaN,NaN
28,ЛР4,classification,ЛР4 случайный лес,sklearn,baseline,0.961111,0.961970,0.960710,0.960722,NaN,NaN,NaN
34,ЛР4,classification,Своя модель лес + параметры CV,собственная реализация,custom_improved,0.958333,0.960252,0.958091,0.958152,NaN,NaN,NaN


Лучшие reference-модели регрессии по RMSE:


,lab,task,model,implementation,stage,accuracy,precision_macro,recall_macro,f1_macro,MAE,RMSE,R2
19,ЛР2,regression,Своя линейная регрессия + новые признаки,собственная реализация,custom_improved,NaN,NaN,NaN,NaN,41.170222,51.891580,0.491759
15,ЛР2,regression,Ridge + новые признаки,sklearn,feature_engineered,NaN,NaN,NaN,NaN,42.105741,52.654356,0.476708
39,ЛР5,regression,ЛР5 гистограммный градиентный бустинг настроенный,sklearn,tuned,NaN,NaN,NaN,NaN,43.056510,53.062205,0.468570
33,ЛР4,regression,Своя модель случайный лес,собственная реализация,custom,NaN,NaN,NaN,NaN,42.494447,53.088426,0.468045
17,ЛР2,regression,Своя линейная регрессия,собственная реализация,custom,NaN,NaN,NaN,NaN,42.794095,53.853446,0.452603
13,ЛР2,regression,Линейная регрессия,sklearn,baseline,NaN,NaN,NaN,NaN,42.794095,53.853446,0.452603
29,ЛР4,regression,ЛР4 случайный лес,sklearn,baseline,NaN,NaN,NaN,NaN,44.053034,54.332408,0.442823
31,ЛР4,regression,ЛР4 случайный лес настроенный,sklearn,tuned,NaN,NaN,NaN,NaN,44.231746,54.490419,0.439577
11,ЛР1,regression,MyKNN улучшенный,собственная реализация,custom_improved,NaN,NaN,NaN,NaN,42.781701,54.493337,0.439517
7,ЛР1,regression,KNN настроенный + стандартизация,sklearn,improved_sklearn,NaN,NaN,NaN,NaN,42.781701,54.493337,0.439517


## Сохранение артефактов и аудит эксперимента

Последняя ячейка формирует CSV/JSON для итогового ноутбука, сохраняет выбранные параметры CV, аудит гипотез, важность признаков и диагностические графики. Для градиентного бустинга использована компактная сетка кандидатов, чтобы ускорить воспроизводимый запуск. Запустите основной ноутбук сверху вниз; после этого можно открыть `final_comparison.ipynb`.


In [27]:
# Формируем сводные CSV/JSON, audit и графики, используемые отдельным итоговым ноутбуком.
metrics_df = pd.DataFrame(rows)
metrics_df.to_csv(OUT / 'metrics.csv', index=False)

reference_stage = {
    'ЛР1': 'improved_sklearn',
    'ЛР2': 'feature_engineered',
    'ЛР3': 'tuned',
    'ЛР4': 'tuned',
    'ЛР5': 'tuned',
}
metric_cols = ['accuracy','precision_macro','recall_macro','f1_macro','MAE','RMSE','R2']

def one_row(frame, lab, task, stage, implementation='sklearn'):
    hit = frame[(frame.lab==lab) & (frame.task==task) & (frame.stage==stage) & (frame.implementation==implementation)]
    if hit.empty:
        return None
    return hit.iloc[0]

def add_metric_values(target, row, suffix):
    for col in metric_cols:
        target[f'{col}_{suffix}'] = None if row is None or col not in row.index or pd.isna(row[col]) else float(row[col])

# Baseline против наиболее релевантного улучшенного sklearn-варианта для каждой пары лаборатория/задача.
baseline_rows=[]
for lab in [f'ЛР{i}' for i in range(1,6)]:
    for task in ['classification','regression']:
        base = one_row(metrics_df,lab,task,'baseline')
        improved = one_row(metrics_df,lab,task,reference_stage[lab])
        rec={'lab':lab,'task':task,'reference_stage':reference_stage[lab],
             'model_baseline':None if base is None else base['model'],
             'model_improved':None if improved is None else improved['model']}
        add_metric_values(rec,base,'baseline'); add_metric_values(rec,improved,'improved')
        for col in metric_cols:
            b=rec.get(f'{col}_baseline'); z=rec.get(f'{col}_improved')
            rec[f'{col}_delta'] = None if b is None or z is None else z-b
        baseline_rows.append(rec)
baseline_vs_tuned = pd.DataFrame(baseline_rows)
baseline_vs_tuned.to_csv(OUT / 'baseline_vs_tuned.csv',index=False)

# Собственные исходные реализации против исходного sklearn baseline.
custom_rows=[]
for lab in [f'ЛР{i}' for i in range(1,6)]:
    for task in ['classification','regression']:
        base=one_row(metrics_df,lab,task,'baseline')
        custom=one_row(metrics_df,lab,task,'custom','собственная реализация')
        rec={'lab':lab,'task':task,
             'custom_model':None if custom is None else custom['model'],
             'baseline_model':None if base is None else base['model']}
        add_metric_values(rec,custom,'custom'); add_metric_values(rec,base,'baseline')
        for col in metric_cols:
            a=rec.get(f'{col}_custom'); b=rec.get(f'{col}_baseline')
            rec[f'{col}_delta']=None if a is None or b is None else a-b
        custom_rows.append(rec)
custom_vs_baseline=pd.DataFrame(custom_rows)
custom_vs_baseline.to_csv(OUT / 'custom_vs_baseline.csv',index=False)

# Улучшенная собственная реализация против improved reference sklearn модели (с указанием конкретной стадии).
custom_improved_rows=[]
for lab in [f'ЛР{i}' for i in range(1,6)]:
    for task in ['classification','regression']:
        ref=one_row(metrics_df,lab,task,reference_stage[lab])
        custom=one_row(metrics_df,lab,task,'custom_improved','собственная реализация')
        rec={'lab':lab,'task':task,'reference_stage':reference_stage[lab],
             'improved_custom_model':None if custom is None else custom['model'],
             'reference_sklearn_model':None if ref is None else ref['model']}
        add_metric_values(rec,custom,'custom_improved'); add_metric_values(rec,ref,'reference_sklearn')
        for col in metric_cols:
            a=rec.get(f'{col}_custom_improved'); b=rec.get(f'{col}_reference_sklearn')
            rec[f'{col}_delta']=None if a is None or b is None else a-b
        custom_improved_rows.append(rec)
custom_improved_vs_reference=pd.DataFrame(custom_improved_rows)
custom_improved_vs_reference.to_csv(OUT / 'custom_improved_vs_tuned.csv',index=False)

# Сохраняем параметры CV и важность признаков случайного леса.
(OUT / 'best_params.json').write_text(json.dumps(best_params,ensure_ascii=False,indent=2),encoding='utf-8')
importance_frames=[]
for task, target_features in [('classification',list(Xc.columns)),('regression',list(Xr.columns))]:
    r=one_row(metrics_df,'ЛР4',task,'tuned')
    if r is not None:
        key=f"ЛР4_{task}_{r['model']}_tuned"
        model=fitted[key]
        if hasattr(model,'feature_importances_'):
            importance_frames.append(pd.DataFrame({'task':task,'feature':target_features,'importance':model.feature_importances_}))
if importance_frames:
    importance_df=pd.concat(importance_frames,ignore_index=True).drop_duplicates(['task','feature'])
else:
    importance_df=pd.DataFrame(columns=['task','feature','importance'])
importance_df['rank']=importance_df.groupby('task')['importance'].rank(ascending=False,method='first')
importance_df.to_csv(OUT / 'feature_importance.csv',index=False)

# Графики диагностики: баланс классов, матрица ошибок, факт/прогноз, loss softmax и важность признаков.
plt.figure(figsize=(8,4))
yc.value_counts().sort_index().plot(kind='bar')
plt.title('Распределение классов Digits')
plt.xlabel('Цифра'); plt.ylabel('Количество объектов'); plt.tight_layout()
plt.savefig(FIG / 'class_distribution.png',dpi=150); plt.close()

candidate_cls=metrics_df[(metrics_df.task=='classification') & (metrics_df.implementation=='sklearn') & metrics_df.stage.isin(reference_stages)]
best_cls=candidate_cls.sort_values('f1_macro',ascending=False).iloc[0]
cls_key=f"{best_cls['lab']}_classification_{best_cls['model']}_{best_cls['stage']}"
cls_pred=predictions[cls_key]
fig,ax=plt.subplots(figsize=(7,6))
ConfusionMatrixDisplay.from_predictions(yc_te,cls_pred,labels=np.sort(yc.unique()),ax=ax,cmap='Blues',colorbar=False)
ax.set_title(f"Матрица ошибок: {best_cls['lab']} / {best_cls['model']}")
fig.tight_layout(); fig.savefig(FIG / 'best_classifier_confusion_matrix.png',dpi=150); plt.close(fig)

candidate_reg=metrics_df[(metrics_df.task=='regression') & (metrics_df.implementation=='sklearn') & metrics_df.stage.isin(reference_stages)]
best_reg=candidate_reg.sort_values('RMSE').iloc[0]
reg_key=f"{best_reg['lab']}_regression_{best_reg['model']}_{best_reg['stage']}"
reg_pred=predictions[reg_key]
fig,ax=plt.subplots(figsize=(6,6))
ax.scatter(yr_te,reg_pred,alpha=.75)
lo=min(float(np.min(yr_te)),float(np.min(reg_pred))); hi=max(float(np.max(yr_te)),float(np.max(reg_pred)))
ax.plot([lo,hi],[lo,hi],'--',linewidth=1)
ax.set_title(f"Факт/прогноз: {best_reg['lab']} / {best_reg['model']}")
ax.set_xlabel('Фактическое значение'); ax.set_ylabel('Прогноз'); fig.tight_layout()
fig.savefig(FIG / 'best_regressor_actual_vs_predicted.png',dpi=150); plt.close(fig)

softmax_base=fitted['ЛР2_classification_Своя softmax-регрессия_custom']
softmax_features=fitted['ЛР2_classification_Своя softmax-регрессия + новые признаки_custom_improved']
plt.figure(figsize=(7,4))
plt.plot(softmax_base.loss_history_,label='Baseline')
plt.plot(softmax_features.loss_history_,label='С новыми признаками')
plt.title('Сходимость собственной softmax-регрессии')
plt.xlabel('Итерация'); plt.ylabel('Loss'); plt.legend(); plt.tight_layout()
plt.savefig(FIG / 'softmax_loss.png',dpi=150); plt.close()

if not importance_df.empty:
    fig,ax=plt.subplots(figsize=(8,5))
    view=importance_df[importance_df.task=='classification'].nsmallest(15,'rank').sort_values('importance')
    ax.barh(view.feature,view.importance)
    ax.set_title('Топ-15 признаков случайного леса (Digits)')
    ax.set_xlabel('Feature importance'); fig.tight_layout()
    fig.savefig(FIG / 'random_forest_feature_importance_digits.png',dpi=150); plt.close(fig)

# Машиночитаемый аудит: гипотезы подтверждаются только по целевым метрикам на holdout.
def val(lab,task,stage,col):
    r=one_row(metrics_df,lab,task,stage)
    return None if r is None or pd.isna(r[col]) else float(r[col])

hypotheses={}
base_f1=val('ЛР1','classification','baseline','f1_macro'); scaled_f1=val('ЛР1','classification','improved_sklearn','f1_macro')
base_rmse=val('ЛР1','regression','baseline','RMSE'); scaled_rmse=val('ЛР1','regression','improved_sklearn','RMSE')
hypotheses['H1_scaling']={'classification_supported':scaled_f1 is not None and scaled_f1>base_f1,
                          'regression_supported':scaled_rmse is not None and scaled_rmse<base_rmse,
                          'classification_f1_delta':scaled_f1-base_f1,
                          'regression_rmse_delta':scaled_rmse-base_rmse}
fe_f1=val('ЛР2','classification','feature_engineered','f1_macro'); fe_rmse=val('ЛР2','regression','feature_engineered','RMSE')
hypotheses['H2_feature_engineering']={'classification_supported':fe_f1>val('ЛР2','classification','baseline','f1_macro'),
                                      'regression_supported':fe_rmse<val('ЛР2','regression','baseline','RMSE'),
                                      'classification_f1_delta':fe_f1-val('ЛР2','classification','baseline','f1_macro'),
                                      'regression_rmse_delta':fe_rmse-val('ЛР2','regression','baseline','RMSE')}
h3={}
for lab in ['ЛР3','ЛР4','ЛР5']:
    vals={}
    for task,col,better in [('classification','f1_macro','higher'),('regression','RMSE','lower')]:
        b=val(lab,task,'baseline',col); t=val(lab,task,'tuned',col)
        vals[task]={'supported':(t>b if better=='higher' else t<b),'baseline':b,'tuned':t,'delta':t-b}
    h3[lab]=vals
hypotheses['H3_tuning']=h3
forest_vs_tree={}
for task,col,better in [('classification','f1_macro','higher'),('regression','RMSE','lower')]:
    tree=val('ЛР3',task,'tuned',col); forest=val('ЛР4',task,'tuned',col)
    forest_vs_tree[task]={'supported':forest>tree if better=='higher' else forest<tree,'tree':tree,'forest':forest,'delta_forest_minus_tree':forest-tree}
hypotheses['H4_ensemble_vs_single_tree']=forest_vs_tree

audit={
    'random_state':RANDOM_STATE,
    'test_size':0.2,
    'cv_folds':5,
    'classification_dataset':'sklearn.datasets.load_digits',
    'regression_dataset':'sklearn.datasets.load_diabetes',
    'train_rows':{'digits':int(len(Xc_tr)),'diabetes':int(len(Xr_tr))},
    'test_rows':{'digits':int(len(Xc_te)),'diabetes':int(len(Xr_te))},
    'metrics_rows':int(len(metrics_df)),
    'reference_best_classifier':{'lab':best_cls['lab'],'model':best_cls['model'],'stage':best_cls['stage'],'f1_macro':float(best_cls['f1_macro'])},
    'reference_best_regressor':{'lab':best_reg['lab'],'model':best_reg['model'],'stage':best_reg['stage'],'RMSE':float(best_reg['RMSE']),'R2':float(best_reg['R2'])},
    'hypotheses':hypotheses,
    'limitations':['Все выводы основаны на одном фиксированном train/test-разбиении.',
                   'Собственные CART/ансамбли — компактные учебные реализации с ограниченным поиском разбиений.']
}
(OUT / 'audit.json').write_text(json.dumps(audit,ensure_ascii=False,indent=2),encoding='utf-8')

print('Сохранены metrics.csv, baseline_vs_tuned.csv, custom_vs_baseline.csv, custom_improved_vs_tuned.csv, best_params.json, audit.json, feature_importance.csv и диагностические графики в results/figures/.')
display(baseline_vs_tuned)
display(custom_vs_baseline)
display(custom_improved_vs_reference)


Сохранены metrics.csv, baseline_vs_tuned.csv, custom_vs_baseline.csv, custom_improved_vs_tuned.csv, best_params.json, audit.json, feature_importance.csv и диагностические графики в results/figures/.


,lab,task,reference_stage,model_baseline,model_improved,accuracy_baseline,precision_macro_baseline,recall_macro_baseline,f1_macro_baseline,MAE_baseline,...,MAE_improved,RMSE_improved,R2_improved,accuracy_delta,precision_macro_delta,recall_macro_delta,f1_macro_delta,MAE_delta,RMSE_delta,R2_delta
0,ЛР1,classification,improved_sklearn,KNN без масштабирования,KNN настроенный + стандартизация,0.983333,0.983449,0.983095,0.983046,NaN,...,NaN,NaN,NaN,-0.019444,-0.018859,-0.019374,-0.019606,NaN,NaN,NaN
1,ЛР1,regression,improved_sklearn,KNN без масштабирования,KNN настроенный + стандартизация,NaN,NaN,NaN,NaN,42.648475,...,42.781701,54.493337,0.439517,NaN,NaN,NaN,NaN,0.133226,-0.159888,0.003294
2,ЛР2,classification,feature_engineered,Логистическая регрессия,Логистическая регрессия + новые признаки,0.958333,0.958731,0.958012,0.958075,NaN,...,NaN,NaN,NaN,0.022222,0.022108,0.022385,0.022355,NaN,NaN,NaN
3,ЛР2,regression,feature_engineered,Линейная регрессия,Ridge + новые признаки,NaN,NaN,NaN,NaN,42.794095,...,42.105741,52.654356,0.476708,NaN,NaN,NaN,NaN,-0.688354,-1.199089,0.024105
4,ЛР3,classification,tuned,ЛР3 решающее дерево,ЛР3 решающее дерево настроенный,0.825000,0.823584,0.824247,0.823049,NaN,...,NaN,NaN,NaN,0.000000,0.000000,0.000000,0.000000,NaN,NaN,NaN
5,ЛР3,regression,tuned,ЛР3 решающее дерево,ЛР3 решающее дерево настроенный,NaN,NaN,NaN,NaN,54.528090,...,48.096592,59.604541,0.329445,NaN,NaN,NaN,NaN,-6.431498,-10.941882,0.268791
6,ЛР4,classification,tuned,ЛР4 случайный лес,ЛР4 случайный лес настроенный,0.961111,0.961970,0.960710,0.960722,NaN,...,NaN,NaN,NaN,0.000000,0.000000,0.000000,0.000000,NaN,NaN,NaN
7,ЛР4,regression,tuned,ЛР4 случайный лес,ЛР4 случайный лес настроенный,NaN,NaN,NaN,NaN,44.053034,...,44.231746,54.490419,0.439577,NaN,NaN,NaN,NaN,0.178713,0.158011,-0.003246
8,ЛР5,classification,tuned,ЛР5 гистограммный градиентный бустинг,ЛР5 гистограммный градиентный бустинг настроенный,0.955556,0.955918,0.955000,0.954733,NaN,...,NaN,NaN,NaN,-0.002778,-0.002327,-0.002778,-0.002493,NaN,NaN,NaN
9,ЛР5,regression,tuned,ЛР5 гистограммный градиентный бустинг,ЛР5 гистограммный градиентный бустинг настроенный,NaN,NaN,NaN,NaN,45.204033,...,43.056510,53.062205,0.468570,NaN,NaN,NaN,NaN,-2.147523,-3.967040,0.082432


,lab,task,custom_model,baseline_model,accuracy_custom,precision_macro_custom,recall_macro_custom,f1_macro_custom,MAE_custom,RMSE_custom,...,MAE_baseline,RMSE_baseline,R2_baseline,accuracy_delta,precision_macro_delta,recall_macro_delta,f1_macro_delta,MAE_delta,RMSE_delta,R2_delta
0,ЛР1,classification,MyKNN,KNN без масштабирования,0.983333,0.983449,0.983095,0.983046,NaN,NaN,...,NaN,NaN,NaN,0.000000,0.000000,0.000000,0.000000,NaN,NaN,NaN
1,ЛР1,regression,MyKNN,KNN без масштабирования,NaN,NaN,NaN,NaN,42.648475,54.653225,...,42.648475,54.653225,0.436223,NaN,NaN,NaN,NaN,0.000000e+00,0.000000,0.000000
2,ЛР2,classification,Своя softmax-регрессия,Логистическая регрессия,0.958333,0.959047,0.958087,0.958137,NaN,NaN,...,NaN,NaN,NaN,0.000000,0.000316,0.000075,0.000061,NaN,NaN,NaN
3,ЛР2,regression,Своя линейная регрессия,Линейная регрессия,NaN,NaN,NaN,NaN,42.794095,53.853446,...,42.794095,53.853446,0.452603,NaN,NaN,NaN,NaN,2.131628e-14,0.000000,0.000000
4,ЛР3,classification,Своя модель решающее дерево,ЛР3 решающее дерево,0.844444,0.848408,0.844314,0.841860,NaN,NaN,...,NaN,NaN,NaN,0.019444,0.024824,0.020066,0.018811,NaN,NaN,NaN
5,ЛР3,regression,Своя модель решающее дерево,ЛР3 решающее дерево,NaN,NaN,NaN,NaN,47.267142,59.069458,...,54.528090,70.546423,0.060654,NaN,NaN,NaN,NaN,-7.260948e+00,-11.476964,0.280777
6,ЛР4,classification,Своя модель случайный лес,ЛР4 случайный лес,0.863889,0.865138,0.863604,0.862041,NaN,NaN,...,NaN,NaN,NaN,-0.097222,-0.096833,-0.097106,-0.098681,NaN,NaN,NaN
7,ЛР4,regression,Своя модель случайный лес,ЛР4 случайный лес,NaN,NaN,NaN,NaN,42.494447,53.088426,...,44.053034,54.332408,0.442823,NaN,NaN,NaN,NaN,-1.558587e+00,-1.243983,0.025222
8,ЛР5,classification,Своя модель градиентный бустинг,ЛР5 гистограммный градиентный бустинг,0.827778,0.831849,0.827104,0.825678,NaN,NaN,...,NaN,NaN,NaN,-0.127778,-0.124069,-0.127896,-0.129055,NaN,NaN,NaN
9,ЛР5,regression,Своя модель градиентный бустинг,ЛР5 гистограммный градиентный бустинг,NaN,NaN,NaN,NaN,46.170385,55.265639,...,45.204033,57.029245,0.386138,NaN,NaN,NaN,NaN,9.663520e-01,-1.763606,0.037380


,lab,task,reference_stage,improved_custom_model,reference_sklearn_model,accuracy_custom_improved,precision_macro_custom_improved,recall_macro_custom_improved,f1_macro_custom_improved,MAE_custom_improved,...,MAE_reference_sklearn,RMSE_reference_sklearn,R2_reference_sklearn,accuracy_delta,precision_macro_delta,recall_macro_delta,f1_macro_delta,MAE_delta,RMSE_delta,R2_delta
0,ЛР1,classification,improved_sklearn,MyKNN улучшенный,KNN настроенный + стандартизация,0.963889,0.964590,0.963722,0.963440,NaN,...,NaN,NaN,NaN,0.000000,0.000000,0.000000,0.000000,NaN,NaN,NaN
1,ЛР1,regression,improved_sklearn,MyKNN улучшенный,KNN настроенный + стандартизация,NaN,NaN,NaN,NaN,42.781701,...,42.781701,54.493337,0.439517,NaN,NaN,NaN,NaN,0.000000,0.000000,0.000000
2,ЛР2,classification,feature_engineered,Своя softmax-регрессия + новые признаки,Логистическая регрессия + новые признаки,0.975000,0.975842,0.974916,0.975020,NaN,...,NaN,NaN,NaN,-0.005556,-0.004998,-0.005480,-0.005411,NaN,NaN,NaN
3,ЛР2,regression,feature_engineered,Своя линейная регрессия + новые признаки,Ridge + новые признаки,NaN,NaN,NaN,NaN,41.170222,...,42.105741,52.654356,0.476708,NaN,NaN,NaN,NaN,-0.935519,-0.762776,0.015052
4,ЛР3,classification,tuned,Своя модель дерево + параметры CV,ЛР3 решающее дерево настроенный,0.763889,0.765827,0.763419,0.762739,NaN,...,NaN,NaN,NaN,-0.061111,-0.057756,-0.060828,-0.060311,NaN,NaN,NaN
5,ЛР3,regression,tuned,Своя модель дерево + параметры CV,ЛР3 решающее дерево настроенный,NaN,NaN,NaN,NaN,47.793953,...,48.096592,59.604541,0.329445,NaN,NaN,NaN,NaN,-0.302639,1.170328,-0.026591
6,ЛР4,classification,tuned,Своя модель лес + параметры CV,ЛР4 случайный лес настроенный,0.958333,0.960252,0.958091,0.958152,NaN,...,NaN,NaN,NaN,-0.002778,-0.001718,-0.002619,-0.002570,NaN,NaN,NaN
7,ЛР4,regression,tuned,Своя модель лес + параметры CV,ЛР4 случайный лес настроенный,NaN,NaN,NaN,NaN,43.947628,...,44.231746,54.490419,0.439577,NaN,NaN,NaN,NaN,-0.284118,0.387737,-0.008004
8,ЛР5,classification,tuned,Своя модель бустинг + параметры CV,ЛР5 гистограммный градиентный бустинг настроенный,0.875000,0.879888,0.874260,0.873458,NaN,...,NaN,NaN,NaN,-0.077778,-0.073703,-0.077962,-0.078782,NaN,NaN,NaN
9,ЛР5,regression,tuned,Своя модель бустинг + параметры CV,ЛР5 гистограммный градиентный бустинг настроенный,NaN,NaN,NaN,NaN,51.448420,...,43.056510,53.062205,0.468570,NaN,NaN,NaN,NaN,8.391910,6.336564,-0.134503
